In [ ]:
# Dependencies: install once from project root, then run all cells. Use the project's .venv kernel.
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
req = root / "requirements.txt"
try:
    import pandas  # noqa: F401
    import xgboost  # noqa: F401
    print("Dependencies OK. Run the next cells.")
except ImportError:
    import subprocess
    import sys
    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
        print("Requirements installed. Restart kernel, then run all cells.")
    else:
        print("From project root run: pip install -r requirements.txt")

# Student Prediction Analytics — A Stakeholder Narrative

This notebook tells the story of a **production-grade ML pipeline** built for the mess of real life: missing exit dates, six partners with different schemas (CRM, SIS, GA4), and sparse joins. One codebase, one config per partner; we record every run and **never drop a lead**. What you see here is the same rigor we bring to stakeholder conversations—accuracy, explainability, fairness, and a clear path from model to action.

**The arc of the narrative:**
1. **Data & design** — Why multi-partner from day one; how we handle missing data and sparse joins.
2. **Features & models** — What we feed the model, how we train it, and how we know it’s trustworthy.
3. **Explainability & fairness** — What drives each prediction; how the model behaves across student groups.
4. **Action** — Coach lists, lead scores 1–100 with reasons, bands, and thresholds so teams know whom to call first.
5. **Delivery** — Scores and reasons in **Salesforce** so enrollment and success see one source of truth.

**How to run:** Use the project's Python environment (e.g. `.venv`) and run cells **top to bottom** so that data load, feature engineering, and train/load all run before the Coach List and Q&A sections. If you see NumPy/scipy import errors, switch the kernel to the project's virtualenv and ensure `pip install -r requirements.txt` has been run there.

---

### 2026 lens: Where this demo aligns with current AI & data science priorities

As of 2026, leaders care about **human–AI collaboration**, **governance & explainability**, **fairness monitoring**, and **data lineage**. This pipeline is built for that:

- **Human-in-the-loop** — Scores and coach lists **augment** advisors; decisions stay with people. The model ranks who to contact first; advisors choose how to intervene.
- **Governance & explainability** — We record every run (config, timestamp, metrics, artifact paths). SHAP and feature importance show *what* drives each prediction so we can explain and audit.
- **Fairness & bias monitoring** — We evaluate model behavior across demographics (Q3) and, in production, would track parity and prediction drift over time so no student group is systematically disadvantaged.
- **Data lineage** — One codebase, config-driven per partner; same feature code for training and scoring so we know which data and logic produced each score.


### Real-life challenges this project was built to tackle

In production we ran into the kinds of issues that break naive ML pipelines. The design below is the direct response to those challenges—so that when stakeholders ask *“What about missing data?”* or *“How do we run this for another school?”*, the answers are already built in.

| Challenge | What we did |
|-----------|-------------|
| **Missing exit dates** | Retention data often has no formal exit date (drops, transfers, data lag). We model this with a configurable `missing_exit_rate`, use **withdrawn** as the target (derived from enrollment state), and add **missing-data indicators** in feature engineering so the model knows when key fields are absent. |
| **Multi-partner (6 universities)** | **Retention wasn't viable with one partner**—too few student-semester records for a stable model. We designed a **single config-driven pipeline** from day one: one codebase, per-partner config for data paths and options. Pooling across partners gave the volume needed to make retention scoring production-ready; same train/score code runs everywhere. |
| **Different sources per partner: CRM, SIS, GA4** | CRM, SIS, and GA4 have different schemas and join keys. We **normalize at merge time**: GA4 as the spine (universal), left-join CRM and SIS with explicit **coverage tracking** (`has_crm_data`, `has_sis_data`) so we can score even when CRM/SIS are missing. |
| **Low GA4→CRM join coverage** | In practice, only a fraction of GA4 leads appear in CRM. We **never drop leads**: left joins keep everyone; missing CRM/SIS is filled and flagged. Coverage is logged every run and monitored. |
| **Scoring across unis & recording runs** | For lifecycle model dev we need to **score across partners** and **record each run** (config, metrics, artifact paths). We persist models and tuned params under `models/`, and optionally log run metadata (timestamp, partner, metrics) for audit and rollback. |

The notebook below shows these choices in the data load, EDA, feature engineering, and training steps.

### Data levels & pipeline scheduling — what can come in and how to run it

Production data arrives at **different granularities**. The pipeline design and schedule depend on which level each system provides and when.

| Level | Description | Typical sources | What we do |
|-------|-------------|-----------------|------------|
| **Event** | One row per user action (page_view, form_submit, click). High volume, high latency if not aggregated. | GA4 BigQuery export, CDP/segment events | **Never** train or score on raw events. Ingest (stream or batch), then **aggregate** to session/user in a separate job. Schedule: near real-time or hourly. |
| **Session** | One row per visit/session. Medium volume; session-level metrics (count of events, duration, bounce). | GA4 sessions, or derived from events (session_id + aggregation). | Aggregate to **user/lead** before joining to CRM/SIS. Use for “sessions in last 7d”, “avg session duration”. Schedule: daily (or hourly if SLA demands). |
| **User / Lead** | One row per person (GA4 user_id, CRM lead_id). Spine for lead scoring; join key for CRM/SIS. | GA4 (after aggregation), CRM (lead table). | **Spine** for lead scoring. Merge CRM and SIS via left join; keep coverage flags. Schedule: after CRM/SIS sync (daily typical). |
| **Enrollment / Student-semester** | One row per enrollment (student × term). Required for retention target (withdrawn) and SIS features. | SIS (enrollment, grades, attendance). | Retention **target** and SIS features live here. Training only after term boundaries or stable snapshots; scoring at mid-term snapshots. |

**Scheduling in practice**

- **Ingestion**: Event-level from GA4 (e.g. daily export to BQ or hourly batch). Idempotent by date/partition; backfill = re-run for date range.
- **Aggregation**: Events → sessions → user-level features. Run **after** ingestion for that date; same idempotency (overwrite partition or upsert by user_id + date).
- **Merge & features**: GA4 spine (user-level) + CRM + SIS. Run **after** CRM and SIS daily loads. Output = one row per lead (and/or per student-semester for retention). Deterministic so re-runs are safe.
- **Training**: Retention = when new term data is stable (e.g. end-of-term or weekly during term). Lead = when enough new conversions exist (e.g. weekly). Heavy job; separate from scoring.
- **Scoring**: Lead scoring = daily (or on-demand) on latest merged table. Retention = at defined snapshots (e.g. mid-term) on enrolled cohort. Use same feature code as training; record run metadata (timestamp, partner, model version) for audit.

This notebook uses **already-aggregated** GA4/CRM/SIS (user- and enrollment-level) so we focus on merge, features, and models. In production, a preceding pipeline would do event → session → user aggregation and then feed these tables.

## Setup — Project path and imports

Run from the project root (the folder that contains `src/` and `config.yaml`), or set `PROJECT_ROOT` accordingly. The next cells load data, config, and models so the rest of the narrative runs end to end.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5)
%matplotlib inline

import seaborn as sns
# Elite viz: refined palette, typography, clean spines
PALETTE = {"primary": "#1e3a5f", "accent": "#c45c26", "teal": "#2e86ab", "coral": "#e07a5f", "success": "#2d6a4f", "danger": "#c1121f"}
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1, rc={"axes.spines.top": False, "axes.spines.right": False})
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "600"
plt.rcParams["figure.titlesize"] = 14
plt.rcParams["figure.titleweight"] = "600"

from src.data_generation import RetentionDataGenerator, LeadScoringDataGenerator, load_config
from src.feature_engineering import RetentionFeatureEngineer, LeadScoringFeatureEngineer
from src.models import RetentionModel, LeadScoringModel
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
from sklearn.calibration import calibration_curve
import joblib
# SHAP imported in Q2 cell where used (avoids numba/numpy constraints at startup)

print(f"Project root: {PROJECT_ROOT}")

## 1. Load or generate data

We use synthetic data that mirrors **real-world patterns**: demographics, **missing exit dates** (config-driven), **sparse GA4→CRM→SIS joins**, and class imbalance. The pipeline is **config-driven**: the same code runs across partners; only `config.yaml` and data paths change. Below we load (or generate) retention and lead data and report the quality metrics we track in production.

In [ ]:
config = load_config()
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(exist_ok=True)

# Retention
ret_path = data_dir / "retention_data.csv"
if ret_path.exists():
    retention_df = pd.read_csv(ret_path)
    retention_df["exit_date"] = pd.to_datetime(retention_df["exit_date"], errors="coerce")
    print(f"Loaded retention: {len(retention_df):,} records")
else:
    retention_df = RetentionDataGenerator(config).generate()
    retention_df.to_csv(ret_path, index=False)
    print(f"Generated retention: {len(retention_df):,} records")

# Lead scoring
ga4_path = data_dir / "ga4_data.csv"
if ga4_path.exists():
    ga4_df = pd.read_csv(ga4_path)
    crm_df = pd.read_csv(data_dir / "crm_data.csv")
    sis_df = pd.read_csv(data_dir / "sis_data.csv")
    print(f"Loaded leads: GA4={len(ga4_df):,}, CRM={len(crm_df):,}, SIS={len(sis_df):,}")
else:
    ga4_df, crm_df, sis_df = LeadScoringDataGenerator(config).generate()
    ga4_df.to_csv(ga4_path, index=False)
    crm_df.to_csv(data_dir / "crm_data.csv", index=False)
    sis_df.to_csv(data_dir / "sis_data.csv", index=False)
    print(f"Generated leads: GA4={len(ga4_df):,}, CRM={len(crm_df):,}, SIS={len(sis_df):,}")

print(f"\nRetention withdrawal rate: {retention_df['withdrawn'].mean():.1%}")
print(f"Lead enrollment rate: {len(sis_df)/len(ga4_df):.1%}")

# Real-life data quality metrics we track every run
withdrawn_rows = retention_df[retention_df["withdrawn"] == 1]
exit_missing = withdrawn_rows["exit_date"].isna().sum()
exit_pct = (exit_missing / len(withdrawn_rows) * 100) if len(withdrawn_rows) else 0
crm_coverage = len(crm_df) / len(ga4_df) * 100
sis_coverage = len(sis_df) / len(ga4_df) * 100
print(f"\n--- Data quality (real-life challenges) ---")
print(f"Missing exit date (among withdrawn): {exit_missing:,} / {len(withdrawn_rows):,} ({exit_pct:.1f}%)")
print(f"GA4→CRM join coverage: {crm_coverage:.1f}%  (GA4 spine; left-join CRM)")
print(f"GA4→SIS join coverage: {sis_coverage:.1f}%  (SIS = enrolled only)")

### Why multi-partner from day one: retention wasn't viable with one partner

**The retention (withdrawal-risk) model needs enough student-semester records** to train reliably and to support calibration, fairness checks, and actionable coach lists. With **a single university partner**, we had too few enrollments per term and too few withdrawals to get a usable model—unstable metrics, poor generalization, and no statistical power for subgroup analysis.

**The design decision: multi-partner config from the start.** Instead of building a one-off pipeline per school, we made the pipeline **config-driven** so the same code runs across N partners. Each partner gets a config (data paths, options); we **pool data** (or train per-partner with shared code). That gave us the volume needed to make the retention model **viable and production-ready**—and the same pattern scales to lead scoring and future products. One codebase, N configs, real-world solution.

In [ ]:
# Visual: single partner vs multi-partner — why retention needed this
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
from matplotlib.patches import FancyBboxPatch
# Left: 1 partner → thin data → not viable
ax.text(0.5, 1.5, "1 partner", fontsize=11, fontweight="700", ha="center")
box = FancyBboxPatch((0.1, 0.5), 1.8, 0.65, boxstyle="round,pad=0.03", facecolor=PALETTE["coral"], edgecolor="white", alpha=0.9)
ax.add_patch(box)
ax.text(1.0, 0.825, "Insufficient\nstudent-semester\nvolume", fontsize=9, ha="center", va="center", color="white", fontweight="600")
ax.annotate("", xy=(2.4, 0.825), xytext=(1.9, 0.825), arrowprops=dict(arrowstyle="->", color="gray", lw=2))
box2 = FancyBboxPatch((2.5, 0.5), 1.6, 0.65, boxstyle="round,pad=0.03", facecolor=PALETTE["danger"], edgecolor="white", alpha=0.9)
ax.add_patch(box2)
ax.text(3.3, 0.825, "Retention model\nnot viable", fontsize=9, ha="center", va="center", color="white", fontweight="600")
# Right: N partners → pooled data → viable
ax.text(5.8, 1.5, "N partners (config-driven)", fontsize=11, fontweight="700", ha="center")
box3 = FancyBboxPatch((4.6, 0.5), 2.0, 0.65, boxstyle="round,pad=0.03", facecolor=PALETTE["teal"], edgecolor="white", alpha=0.9)
ax.add_patch(box3)
ax.text(5.6, 0.825, "Pooled data,\nsame code & config", fontsize=9, ha="center", va="center", color="white", fontweight="600")
ax.annotate("", xy=(7.5, 0.825), xytext=(6.6, 0.825), arrowprops=dict(arrowstyle="->", color="gray", lw=2))
box4 = FancyBboxPatch((7.6, 0.5), 1.9, 0.65, boxstyle="round,pad=0.03", facecolor=PALETTE["success"], edgecolor="white", alpha=0.9)
ax.add_patch(box4)
ax.text(8.55, 0.825, "Viable, production-\nready retention model", fontsize=9, ha="center", va="center", color="white", fontweight="600")
ax.set_title("Single partner vs multi-partner: why we designed for N partners from day one", fontsize=12, fontweight="600", pad=12)
plt.tight_layout()
plt.show()

**Multi-partner config & run recording (lifecycle model dev)**

One codebase, **config-driven** per partner (or per run). We score across all universities by pointing at the right data and config; **run recording** = persisting models, tuned params, and optional run metadata so we can audit and roll back.

In [ ]:
# Config-driven pipeline: one config per partner/run
import json
from datetime import datetime, timezone
paths = config.get("paths", {})
data_dir = PROJECT_ROOT / paths.get("data_dir", "data")
models_dir = PROJECT_ROOT / paths.get("models_dir", "models")
outputs_dir = PROJECT_ROOT / paths.get("outputs_dir", "outputs")
for d in (models_dir, outputs_dir):
    d.mkdir(parents=True, exist_ok=True)
print("Config (data/model/paths) drives the pipeline; change config + data to run for another partner.\n")
print("Run artifacts we persist for every train run:")
print(f"  - {models_dir}/  (retention_early_model.pkl, retention_mid_model.pkl, lead_scoring_model.pkl)")
print(f"  - {models_dir}/tuned_params.json  (Optuna-tuned hyperparameters)")
tuned_path = models_dir / "tuned_params.json"
if tuned_path.exists():
    with open(tuned_path) as f:
        tuned = json.load(f)
    print(f"  - tuned_params keys: {list(tuned.keys())}")
# Optional: log this run for audit (partner_id could come from config in production)
run_log = {"timestamp": datetime.now(timezone.utc).isoformat(), "config_data": config.get("data", {})}
run_log_path = outputs_dir / "last_run_metadata.json"
with open(run_log_path, "w") as f:
    json.dump(run_log, f, indent=2)
print(f"  - {run_log_path}  (run metadata for audit)")

### Visual: Multi-partner config & pipeline scheduling

Impress stakeholders with a clear picture of **one codebase, N partners** and **when each pipeline stage runs**. Below: (1) config-driven flow from codebase to artifacts, (2) scheduling cadence so everyone sees ingestion → aggregation → merge → score vs. train.

In [ ]:
# Two-panel visual: multi-partner config flow + pipeline scheduling
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5))

# --- Panel 1: One codebase, N configs, same pipeline ---
ax1 = axes[0]
ax1.set_xlim(0, 10); ax1.set_ylim(0, 3); ax1.set_aspect("equal"); ax1.axis("off")
# Boxes: Codebase -> Configs -> Data -> Train/Score -> Artifacts
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, label, color=PALETTE["teal"]):
    p = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", facecolor=color, edgecolor="white", linewidth=1.2, alpha=0.9)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9, fontweight="600", color="white")
box(ax1, 0.2, 1, 1.4, 0.7, "One\ncodebase", PALETTE["primary"])
box(ax1, 2.0, 1, 1.6, 0.7, "Config per partner\n(A, B, C, ...)", PALETTE["accent"])
box(ax1, 4.0, 1, 1.2, 0.7, "Data\npaths", PALETTE["teal"])
box(ax1, 5.6, 1, 1.5, 0.7, "Train / Score", PALETTE["coral"])
box(ax1, 7.5, 1, 1.8, 0.7, "Models, tuned_params,\nrun_metadata", PALETTE["success"])
for x in [1.6, 3.6, 5.2, 7.1]:
    ax1.annotate("", xy=(x + 0.35, 1.35), xytext=(x, 1.35), arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
ax1.set_title("Multi-partner: one codebase, config + data per partner, same pipeline", fontsize=11, fontweight="600", pad=10)

# --- Panel 2: Scheduling cadence ---
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 5.5); ax2.axis("off")
# (label, y_pos, cadence, bar_length for visual)
stages = [
    ("Event ingest (GA4)", 4.2, "hourly", 2.2),
    ("Aggregate (events → sessions → user)", 3.4, "daily", 1.6),
    ("Merge (GA4 + CRM + SIS)", 2.6, "daily", 1.6),
    ("Score (all leads)", 1.8, "daily", 1.6),
    ("Train (retention / lead)", 1.0, "weekly", 1.0),
]
ax2.text(0.2, 4.85, "Pipeline stage", fontsize=9, fontweight="600")
ax2.text(5.2, 4.85, "Cadence", fontsize=9, fontweight="600")
for label, y, cadence, bar_w in stages:
    ax2.text(0.2, y, label, fontsize=9, va="center")
    color = PALETTE["accent"] if "Train" in label else PALETTE["teal"]
    p = FancyBboxPatch((4.2, y - 0.2), bar_w, 0.4, boxstyle="round,pad=0.02", facecolor=color, edgecolor="white", alpha=0.85)
    ax2.add_patch(p)
    ax2.text(4.2 + bar_w/2, y, cadence, ha="center", va="center", fontsize=8, color="white", fontweight="600")
ax2.set_title("Scheduling: when each step runs (idempotent by date; backfill = re-run for date range)", fontsize=11, fontweight="600", pad=10)
plt.tight_layout()
plt.show()

## 2. EDA — A stakeholder snapshot

*What does the data look like, and where are the quality issues?* The plots below are styled so they can drop straight into decks and reports.

In [ ]:
# Retention: key distributions — publication-style
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
first_sem = retention_df[retention_df["semester"] == 1]

sns.histplot(first_sem["gpa_high_school"], bins=30, color=PALETTE["teal"], edgecolor="white", linewidth=0.5, ax=axes[0,0])
axes[0,0].set_title("HS GPA (First Semester)")
axes[0,0].set_xlabel("GPA")

sns.histplot(first_sem["risk_score"], bins=20, color=PALETTE["coral"], edgecolor="white", linewidth=0.5, ax=axes[0,1])
axes[0,1].set_title("Pre-computed Risk Score")
axes[0,1].set_xlabel("Risk")

withdrawn_by_sem = retention_df.groupby("semester").agg(
    students=("student_id", "nunique"),
    withdrawn=("withdrawn", "sum")
).reset_index()
withdrawn_by_sem["rate"] = withdrawn_by_sem["withdrawn"] / withdrawn_by_sem["students"] * 100
bars0 = axes[1,0].bar(withdrawn_by_sem["semester"], withdrawn_by_sem["rate"], color=PALETTE["teal"], edgecolor="white", linewidth=0.5)
for b, r in zip(bars0, withdrawn_by_sem["rate"]):
    axes[1,0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{r:.1f}%", ha="center", va="bottom", fontsize=9)
axes[1,0].set_title("Withdrawal Rate by Semester")
axes[1,0].set_xlabel("Semester")
axes[1,0].set_ylabel("% Withdrawn")

missing_exit = retention_df[retention_df["withdrawn"]==1]
missing_pct = missing_exit["exit_date"].isna().mean() * 100 if len(missing_exit) > 0 else 0
vals = [100 - missing_pct, missing_pct]
bars1 = axes[1,1].bar(["Has Exit Date", "Missing Exit Date"], vals, color=[PALETTE["success"], PALETTE["danger"]], edgecolor="white", linewidth=0.5)
for b, v in zip(bars1, vals):
    axes[1,1].text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)
axes[1,1].set_title("Data Quality: Missing Exit Dates (Withdrawn Only)")
axes[1,1].set_ylabel("%")
for ax in axes.flat:
    sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.suptitle("Retention Data — Key Distributions & Quality Issues", y=1.02, fontsize=15, fontweight="600")
plt.show()

**Data quality in the wild — how we handle it**

- **Missing exit dates:** We don't rely on `exit_date` for the target; we use **withdrawn** (enrollment state). Config sets `missing_exit_rate` so synthetic data matches reality. In feature engineering we add **missing-data indicators** (e.g. `missing_early_attendance`) so the model sees when key fields are absent.
- **Low GA4→CRM join coverage:** We keep **GA4 as the spine** and left-join CRM and SIS. No lead is dropped. We add **has_crm_data** and **has_sis_data** flags and fill missing CRM/SIS with medians/modes so we can score every lead across partners.

## Actionable insights: coach list (at-risk students per school)

To turn predictions into **early intervention**, we build a **coach list**: the most at-risk students per school, with **reasons** (e.g. low mid-GPA, attendance warning, financial stress). Coaches see whom to call first and why—before students exit.

In [ ]:
from src.insights import build_coach_list
from src.feature_engineering import RetentionFeatureEngineer
from src.models import RetentionModel

# Build mid_feat and load model if not in scope (run after Feature Engineering + Train, or we create/load here)
_g = globals()
if "mid_feat" not in _g or "mid_cols" not in _g:
    fe = RetentionFeatureEngineer()
    mid_feat = fe.create_mid_semester_features(retention_df)
    mid_cols = fe.get_feature_columns("mid")
if "mid_model" not in _g:
    mid_model = RetentionModel.load(str(PROJECT_ROOT / "models" / "retention_mid_model.pkl"))
if "school_id" not in retention_df.columns:
    retention_df["school_id"] = 0
school_names = config.get("data", {}).get("retention", {}).get("school_names", None)

X_full = mid_feat[mid_cols].fillna(mid_feat[mid_cols].median()).fillna(0)
all_risk_probs = mid_model.predict(X_full)
coach_list = build_coach_list(
    retention_df, mid_feat, all_risk_probs,
    school_names=school_names,
    top_n=30,
)

# Display per-school summary and sample
for school in coach_list[:3]:
    print(f"\n{school['school_name']} — {school['count']} at-risk students (top 30)")
    display_df = pd.DataFrame(school["students"][:8])
    display_df["reasons"] = display_df["reasons"].apply(lambda x: "; ".join(x[:3]) if x else "")
    try:
        display(display_df[["student_id", "risk_prob", "band", "band_label", "mid_gpa", "reasons"]])
    except NameError:
        print(display_df[["student_id", "risk_prob", "band", "band_label", "mid_gpa", "reasons"]].to_string())

### Top reasons students are at risk (aggregated from coach list)

What’s actually driving at-risk flags? We aggregate all reasons across the coach list to show which indicators appear most often.

In [ ]:
from collections import Counter

reason_counts = Counter()
for school in coach_list:
    for s in school["students"]:
        for r in (s.get("reasons") or []):
            reason_counts[r] += 1
top_reasons = reason_counts.most_common(12)
labels = [r[0] for r in top_reasons]
counts = [r[1] for r in top_reasons]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(labels))
bars = ax.barh(y_pos, counts, color=PALETTE["coral"], edgecolor="white", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Number of at-risk students with this reason")
ax.set_title("Top reasons students are at risk (from coach list)")
for i, (c, lab) in enumerate(zip(counts, labels)):
    ax.text(c + 2, i, f"{c}", va="center", fontsize=9, fontweight="500")
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate: same pipeline works with sparse joins (left-join, coverage flags)
fe_lead = LeadScoringFeatureEngineer()
merged_lead = fe_lead.merge_sources(ga4_df, crm_df, sis_df)
lead_feat = fe_lead.create_features(merged_lead)
print("Lead scoring: coverage flags after merge (we score everyone)")
print(lead_feat[["has_crm_data", "has_sis_data"]].value_counts().sort_index().to_string())
print(f"\nConfig: missing_exit_rate = {config.get('data', {}).get('retention', {}).get('missing_exit_rate', 'N/A')}")

## 3. Feature Engineering — Why These Features?

**"How did you choose what to feed the model? What about missing data?"**

- **Early-semester (15 features):** Demographics, academic prep, financial indicators, early engagement. Available at semester start.
- **Mid-semester (24 features):** Adds mid-GPA, engagement trends, support utilization, warning flags. Requires ~6 weeks of term.
- **Lead scoring (29 features):** GA4 engagement, CRM marketing touches, SIS academic data. Handles 30% missing CRM, 85% missing SIS (non-enrolled).

In [ ]:
fe = RetentionFeatureEngineer()
early_feat = fe.create_early_semester_features(retention_df)
mid_feat = fe.create_mid_semester_features(retention_df)
early_cols = fe.get_feature_columns("early")
mid_cols = fe.get_feature_columns("mid")

print("Early-semester features (15):", early_cols)
print("\nAdditional mid-semester features (+9):", [c for c in mid_cols if c not in early_cols])

## 4. Train or Load Models

We train if models don't exist; otherwise load from disk.

## What's feeding the model? — Feature importance

Stakeholders ask: **"What's driving the predictions?"** The top features the retention and lead models use (from the trained trees) are below; percentages are relative importance and align with SHAP.

In [ ]:
# Retention: feature importance (from trained or loaded mid-semester model)
if "mid_model" in dir() and getattr(mid_model, "model", None) is not None:
    ret_fi = dict(zip(mid_model.feature_names, mid_model.model.feature_importances_))
elif "mid_results" in dir():
    ret_fi = mid_results.get("feature_importance", {})
else:
    ret_fi = {}
ret_sorted = sorted(ret_fi.items(), key=lambda x: -x[1])[:14]
total = sum(v for _, v in ret_sorted) or 1
labels_ret = [n.replace("_", " ").title() for n, _ in ret_sorted]
pcts_ret = [v / total * 100 for _, v in ret_sorted]

# Lead: if we have lead_model (from later in notebook), show its importance too
lead_fi = {}
if "lead_model" in dir() and getattr(lead_model, "model", None) is not None:
    lead_fi = dict(zip(lead_model.feature_names, lead_model.model.feature_importances_))
lead_sorted = sorted(lead_fi.items(), key=lambda x: -x[1])[:14] if lead_fi else []
total_lead = sum(v for _, v in lead_sorted) or 1
labels_lead = [n.replace("_", " ").title() for n, _ in lead_sorted]
pcts_lead = [v / total_lead * 100 for _, v in lead_sorted]

n_plots = 2 if lead_sorted else 1
fig, axes = plt.subplots(1, n_plots, figsize=(14 if n_plots == 2 else 8, 6))
if n_plots == 1:
    axes = [axes]
y_pos = np.arange(len(labels_ret))[::-1]
axes[0].barh(y_pos, pcts_ret, color=PALETTE["teal"], edgecolor="white", linewidth=0.5)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels_ret, fontsize=10)
axes[0].set_xlabel("Relative importance (%)")
axes[0].set_title("Retention (mid-semester) — What's feeding the model?")
for i, p in enumerate(pcts_ret):
    axes[0].text(p + 0.3, len(labels_ret) - 1 - i, f"{p:.1f}%", va="center", fontsize=9)
sns.despine(trim=True, ax=axes[0])
if lead_sorted:
    y_pos_l = np.arange(len(labels_lead))[::-1]
    axes[1].barh(y_pos_l, pcts_lead, color=PALETTE["accent"], edgecolor="white", linewidth=0.5)
    axes[1].set_yticks(y_pos_l)
    axes[1].set_yticklabels(labels_lead, fontsize=10)
    axes[1].set_xlabel("Relative importance (%)")
    axes[1].set_title("Lead scoring — What's feeding the model?")
    for i, p in enumerate(pcts_lead):
        axes[1].text(p + 0.3, len(labels_lead) - 1 - i, f"{p:.1f}%", va="center", fontsize=9)
    sns.despine(trim=True, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(exist_ok=True)
fe_lead = LeadScoringFeatureEngineer()
merged = fe_lead.merge_sources(ga4_df, crm_df, sis_df)
lead_feat = fe_lead.create_features(merged)
lead_cols = fe_lead.get_feature_columns()
X_lead = lead_feat[lead_cols].fillna(lead_feat[lead_cols].median()).fillna(0)
y_lead = lead_feat["enrolled"]

# Train or load retention
early_path = models_dir / "retention_early_model.pkl"
if early_path.exists():
    early_model = RetentionModel.load(str(early_path))
    early_results = None  # load metrics from artifact if needed
else:
    X_early = early_feat[early_cols]
    early_model = RetentionModel(feature_set="early")
    early_results = early_model.train(X_early, early_feat["withdrawn"])
    early_model.save(str(early_path))

# Mid-semester
mid_path = models_dir / "retention_mid_model.pkl"
if mid_path.exists():
    mid_model = RetentionModel.load(str(mid_path))
    mid_results = None
else:
    X_mid = mid_feat[mid_cols]
    mid_model = RetentionModel(feature_set="mid")
    mid_results = mid_model.train(X_mid, mid_feat["withdrawn"])
    mid_model.save(str(mid_path))

# Lead scoring
lead_path = models_dir / "lead_scoring_model.pkl"
if lead_path.exists():
    lead_model = LeadScoringModel.load(str(lead_path))
    lead_results = None
else:
    lead_model = LeadScoringModel()
    lead_results = lead_model.train(X_lead, y_lead)
    lead_model.save(str(lead_path))

# Get holdout predictions for Q1 (calibration, confusion matrix)
if mid_results is None:
    from sklearn.model_selection import train_test_split
    X_mid = mid_feat[mid_cols]
    _, X_te, _, y_te = train_test_split(X_mid, mid_feat["withdrawn"], test_size=0.2, random_state=42, stratify=mid_feat["withdrawn"])
    mid_results = {"y_test": y_te.values, "y_pred_proba": mid_model.predict(X_te)}

print("Models loaded/trained.")
print(f"Retention Mid AUC: {roc_auc_score(mid_results['y_test'], mid_results['y_pred_proba']):.3f}")

---
# STAKEHOLDER Q&A — Design Choices and Validation

Stakeholders often ask the following; we document how the pipeline is designed and validated.

---

## Q1: "How accurate is this model? What's our error rate?"

We demonstrate: AUC, precision-recall curve, confusion matrix, and **calibration** so predicted probabilities are trustworthy.

In [ ]:
# Use mid-semester retention (best performer) for demo
y_test = np.asarray(mid_results.get("y_test"))
y_pred_proba = np.asarray(mid_results.get("y_pred_proba"))

# Bootstrap 95% CI for AUC (200 resamples)
np.random.seed(42)
aucs = []
for _ in range(200):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    aucs.append(roc_auc_score(y_test[idx], y_pred_proba[idx]))
auc_point = roc_auc_score(y_test, y_pred_proba)
auc_lo, auc_hi = np.percentile(aucs, [2.5, 97.5])

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ROC — fill under curve, elite styling
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0,0].fill_between(fpr, tpr, alpha=0.3, color=PALETTE["teal"])
axes[0,0].plot(fpr, tpr, lw=2.5, color=PALETTE["teal"], label=f"AUC = {auc_point:.3f} (95% CI: [{auc_lo:.3f}, {auc_hi:.3f}])")
axes[0,0].plot([0,1],[0,1], "k--", alpha=0.6)
axes[0,0].set_xlabel("False Positive Rate"); axes[0,0].set_ylabel("True Positive Rate")
axes[0,0].set_title("ROC Curve — Model Discrimination")
axes[0,0].legend(loc="lower right"); axes[0,0].grid(True, alpha=0.3)
sns.despine(trim=True, ax=axes[0,0])

# Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)
axes[0,1].fill_between(recall, precision, alpha=0.3, color=PALETTE["accent"])
axes[0,1].plot(recall, precision, lw=2.5, color=PALETTE["accent"], label=f"AP = {ap:.3f}")
axes[0,1].set_xlabel("Recall"); axes[0,1].set_ylabel("Precision")
axes[0,1].set_title("Precision-Recall — Important for Imbalanced Data")
axes[0,1].legend(loc="upper right"); axes[0,1].grid(True, alpha=0.3)
sns.despine(trim=True, ax=axes[0,1])

# Confusion matrix (0.5 threshold)
y_pred = (y_pred_proba > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
im = axes[1,0].imshow(cm, cmap="Blues", aspect="auto", vmin=0, vmax=cm.max())
axes[1,0].set_xticks([0,1]); axes[1,0].set_yticks([0,1])
axes[1,0].set_xticklabels(["Retained","Withdrawn"]); axes[1,0].set_yticklabels(["Retained","Withdrawn"])
axes[1,0].set_xlabel("Predicted"); axes[1,0].set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        axes[1,0].text(j, i, int(cm[i,j]), ha="center", va="center", fontsize=14, color="white" if cm[i,j] > cm.max()/2 else "black", fontweight="600")
axes[1,0].set_title("Confusion Matrix (threshold=0.5)")
sns.despine(trim=True, ax=axes[1,0])

# Calibration: do predicted probs match actual outcome rates?
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
axes[1,1].plot(prob_pred, prob_true, "o-", lw=2, markersize=8, color=PALETTE["teal"], label="Model")
axes[1,1].plot([0,1],[0,1], "k--", alpha=0.6, label="Perfect")
axes[1,1].set_xlabel("Mean Predicted Probability"); axes[1,1].set_ylabel("Fraction of Positives")
axes[1,1].set_title("Calibration — Can We Trust the Score?")
axes[1,1].legend(loc="upper left"); axes[1,1].grid(True, alpha=0.3)
sns.despine(trim=True, ax=axes[1,1])

plt.suptitle("Q1: How Accurate? Error Rates & Calibration", y=1.02, fontsize=15, fontweight="600")
plt.tight_layout()
plt.show()

# Summary stats
print(classification_report(y_test, y_pred, target_names=["Retained","Withdrawn"]))

### Elite insight: Does the model separate risk? Lift, deciles, and separation

**Stakeholders need to see that the score is actionable.** Three views make the case:

1. **Lift curve** — If we contact students in order of predicted risk (top 20% first), we capture a much larger share of actual withdrawals than random. The higher the curve above the diagonal, the better the model ranks.
2. **Actual withdrawal rate by risk decile** — Students in the top risk decile actually withdraw at a much higher rate; the bottom decile stays. This is calibration and validity in one chart.
3. **Score distribution: Retained vs Withdrawn** — The model assigns higher risk to those who actually leave. Clear separation between the two densities means the score discriminates.

In [ ]:
# Elite viz: lift curve, actual rate by decile, density separation
yt = np.asarray(mid_results.get("y_test"))
yp = np.asarray(mid_results.get("y_pred_proba"))
n = len(yt)
n_pos = int(yt.sum())
order = np.argsort(-yp)
yt_ordered = yt[order]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# 1. Cumulative gains (lift) curve
pct_pop = np.arange(1, n + 1) / n * 100
cum_captured = np.cumsum(yt_ordered)
pct_withdrawals_captured = cum_captured / n_pos * 100 if n_pos > 0 else np.zeros(n)
axes[0].fill_between(pct_pop, pct_withdrawals_captured, alpha=0.35, color=PALETTE["teal"])
axes[0].plot(pct_pop, pct_withdrawals_captured, lw=2.5, color=PALETTE["teal"], label="Model (by risk)")
axes[0].plot([0, 100], [0, 100], "k--", alpha=0.6, label="Random")
axes[0].set_xlabel("% of cohort contacted (by risk rank)")
axes[0].set_ylabel("% of all withdrawals captured")
axes[0].set_title("Lift: contacting by risk captures withdrawals fast")
axes[0].legend(loc="lower right")
axes[0].set_xlim(0, 100); axes[0].set_ylim(0, 100)
axes[0].grid(True, alpha=0.3)
sns.despine(trim=True, ax=axes[0])

# 2. Actual withdrawal rate by predicted risk decile
deciles = pd.qcut(yp, q=10, labels=False, duplicates="drop") + 1
decile_agg = pd.DataFrame({"y": yt, "decile": deciles}).groupby("decile").agg(
    rate=("y", "mean"), count=("y", "count")
).reset_index()
decile_agg["rate_pct"] = decile_agg["rate"] * 100
colors_dec = [PALETTE["danger"] if r > 0.5 else PALETTE["teal"] for r in decile_agg["rate"]]
bars = axes[1].bar(decile_agg["decile"], decile_agg["rate_pct"], color=colors_dec, edgecolor="white", linewidth=0.5)
for r, row in decile_agg.iterrows():
    axes[1].text(row["decile"], row["rate_pct"] + 1.5, f'{row["rate_pct"]:.0f}%', ha="center", fontsize=9, fontweight="500")
axes[1].axhline(yt.mean() * 100, color="gray", linestyle="--", alpha=0.8, label="Overall rate")
axes[1].set_xlabel("Predicted risk decile (1=lowest, 10=highest)")
axes[1].set_ylabel("Actual withdrawal %")
axes[1].set_title("Actual outcome by decile: high risk → high withdrawal")
axes[1].legend()
sns.despine(trim=True, ax=axes[1])

# 3. Density: predicted risk for Retained vs Withdrawn
retained = yp[yt == 0]
withdrawn = yp[yt == 1]
axes[2].hist(retained, bins=25, density=True, alpha=0.6, color=PALETTE["success"], label=f"Retained (n={len(retained)})", edgecolor="white", linewidth=0.5)
axes[2].hist(withdrawn, bins=25, density=True, alpha=0.6, color=PALETTE["danger"], label=f"Withdrawn (n={len(withdrawn)})", edgecolor="white", linewidth=0.5)
axes[2].set_xlabel("Predicted withdrawal probability")
axes[2].set_ylabel("Density")
axes[2].set_title("Score separation: withdrawn students get higher risk")
axes[2].legend()
sns.despine(trim=True, ax=axes[2])

plt.suptitle("Elite insight: model rank and separation", y=1.02, fontsize=14, fontweight="600")
plt.tight_layout()
plt.show()


**Risk deciles & lift** — Binning predicted risk into deciles shows where the model concentrates withdrawals; cumulative gains help prioritize interventions.

In [ ]:
# Risk deciles: actual withdrawal rate and cumulative % of withdrawals by predicted risk
deciles = pd.qcut(y_pred_proba, q=10, labels=False, duplicates="drop") + 1
decile_df = pd.DataFrame({"y_true": y_test, "y_pred_proba": y_pred_proba, "decile": deciles})
rate_by_decile = decile_df.groupby("decile").agg(
    withdrawal_rate=("y_true", "mean"),
    pct_withdrawals=("y_true", lambda s: s.sum() / y_test.sum() * 100),
    count=("y_true", "count"),
).reset_index()
cumulative_pct = rate_by_decile["pct_withdrawals"].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax1, ax2 = axes
x = rate_by_decile["decile"].astype(int)
rates_pct = rate_by_decile["withdrawal_rate"] * 100
bars = ax1.bar(x - 0.35, rates_pct, width=0.7, color=PALETTE["teal"], edgecolor="white", linewidth=0.5)
for b, r in zip(bars, rates_pct):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5, f"{r:.1f}%", ha="center", va="bottom", fontsize=9)
ax1.set_xlabel("Risk decile (1=lowest, 10=highest)")
ax1.set_ylabel("Actual withdrawal rate (%)")
ax1.set_title("Withdrawal rate by predicted risk decile")
ax1.set_xticks(x)
sns.despine(trim=True, ax=ax1)

ax2.plot(x, cumulative_pct, marker="o", color=PALETTE["accent"], lw=2.5, markersize=8)
ax2.fill_between(x, 0, cumulative_pct, alpha=0.25, color=PALETTE["accent"])
ax2.axhline(50, color="gray", linestyle="--", alpha=0.7)
ax2.set_xlabel("Risk decile (cumulative)")
ax2.set_ylabel("Cumulative % of all withdrawals")
ax2.set_title("Cumulative gains — top deciles capture most withdrawals")
ax2.set_xticks(x)
sns.despine(trim=True, ax=ax2)
plt.tight_layout()
plt.show()

## Q2: "Why did you choose these features? What drives the prediction?"

**SHAP values** quantify each feature's contribution. Positive = pushes toward dropout; negative = toward retention. We use this to validate that drivers match domain expectations.

In [ ]:
# SHAP summary — feature importance (business-interpretable)
try:
    import shap
    X_mid = mid_feat[mid_cols].fillna(mid_feat[mid_cols].median())
    sample = X_mid.sample(min(200, len(X_mid)), random_state=42)
    explainer = shap.TreeExplainer(mid_model.model)
    shap_vals = explainer.shap_values(sample)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, sample, max_display=12, show=False)
    plt.title("Q2: What Drives the Model? (SHAP — Red=Higher Risk)", fontsize=13, fontweight="600")
    plt.tight_layout()
    sns.despine(trim=True)
    plt.show()
except ImportError:
    explainer = None
    print("SHAP not available (e.g. NumPy 2.3+ with current Numba). Use env with numpy<2.3 for SHAP plots.")
    print("Feature importance from model (no SHAP):", dict(sorted(zip(mid_model.feature_names, mid_model.model.feature_importances_), key=lambda x: -x[1])[:10]))

## Q3: "Does this disadvantage any student groups? (Fairness)"

We evaluate predicted risk and actual outcomes across **first-gen** and **financial aid** — groups often over-represented among at-risk populations — to ensure the model behaves consistently.

In [ ]:
# Fairness: predicted risk and actual outcome by demographic
first_sem = mid_feat[mid_feat["semester"]==1].copy()
first_sem["pred_risk"] = mid_model.predict(first_sem[mid_cols].fillna(first_sem[mid_cols].median()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fg, label in [(0, "Not First-Gen"), (1, "First-Gen")]:
    subset = first_sem[first_sem["first_gen"]==fg]
    color = PALETTE["teal"] if fg == 0 else PALETTE["coral"]
    axes[0].hist(subset["pred_risk"], bins=20, alpha=0.7, label=f"{label} (n={len(subset)})", density=True, color=color, edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Predicted Withdrawal Risk")
axes[0].set_ylabel("Density")
axes[0].set_title("Predicted Risk Distribution by First-Gen Status")
axes[0].legend()
sns.despine(trim=True, ax=axes[0])

groups = [(0, "Not First-Gen"), (1, "First-Gen")]
x = np.arange(len(groups))
width = 0.35
actual_rates = [first_sem[first_sem["first_gen"]==g]["withdrawn"].mean()*100 for g, _ in groups]
pred_rates = [first_sem[first_sem["first_gen"]==g]["pred_risk"].mean()*100 for g, _ in groups]
axes[1].bar(x - width/2, actual_rates, width, label="Actual % Withdrawn", color=PALETTE["teal"], edgecolor="white", linewidth=0.5)
axes[1].bar(x + width/2, pred_rates, width, label="Avg Predicted Risk %", color=PALETTE["coral"], alpha=0.9, edgecolor="white", linewidth=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels([l for _, l in groups])
axes[1].set_ylabel("%"); axes[1].set_title("Actual vs Predicted by First-Gen")
axes[1].legend()
sns.despine(trim=True, ax=axes[1])
plt.tight_layout()
plt.suptitle("Q3: Fairness — Model Behavior Across Demographics", y=1.02, fontsize=15, fontweight="600")
plt.show()

## Q4: "What's the cost of being wrong? Should we change the threshold?"

**False Positive (FP):** We intervene but student would have stayed — wasted effort.  
**False Negative (FN):** We don't intervene but student withdraws — lost student.

Different thresholds trade off FP vs FN. We report precision and recall at multiple cutoffs so the business can set the operating point.

In [ ]:
y_test = mid_results["y_test"]
y_pred_proba = mid_results["y_pred_proba"]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for thresh in thresholds:
    pred = (y_pred_proba >= thresh).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    rows.append({
        "Threshold": thresh,
        "FP (unnecessary interventions)": fp,
        "FN (missed at-risk)": fn,
        "TP (correctly flagged)": tp,
        "Precision": f"{prec:.2f}",
        "Recall": f"{rec:.2f}"
    })

pd.DataFrame(rows).style.set_caption("Q4: Trade-offs at Different Thresholds — Choose Based on Your Cost of FP vs FN")

## Q5: "Show me a concrete student you'd flag — and why"

We select a high-risk student and demonstrate the **SHAP contributions** — each feature's push toward or against dropout — so advisors see exactly why that student is flagged.

In [ ]:
# Pick highest-risk student and show top SHAP contributions (or feature importance fallback)
high_risk_idx = first_sem["pred_risk"].idxmax()
student_row = mid_feat.loc[[high_risk_idx]][mid_cols].fillna(mid_feat[mid_cols].median()).fillna(0)
pred_risk = first_sem.loc[high_risk_idx, "pred_risk"]

if "explainer" in dir() and explainer is not None:
    single_shap = explainer.shap_values(student_row)
    shap_series = pd.Series(single_shap[0], index=student_row.columns)
    contrib = shap_series.reindex(shap_series.abs().sort_values(ascending=False).index).head(8)
else:
    contrib = pd.Series(dict(zip(mid_model.feature_names, mid_model.model.feature_importances_))).sort_values(ascending=False).head(8)

colors = [PALETTE["danger"] if v > 0 else PALETTE["success"] for v in contrib.values]
plt.figure(figsize=(10, 5))
plt.barh(contrib.index, contrib.values, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("SHAP value (contribution to prediction)" if "explainer" in dir() and explainer is not None else "Feature importance")
plt.title(f"Q5: Why is this student high-risk? Predicted withdrawal probability: {pred_risk:.2f}", fontweight="600")
plt.gca().invert_yaxis()
sns.despine(trim=True)
plt.tight_layout()
plt.show()

# Show actual student profile
print("Student profile (key fields):")
prof = first_sem.loc[high_risk_idx, ["gpa_high_school", "sat_score", "first_gen", "financial_aid", "part_time", 
                                     "mid_gpa", "mid_attendance", "gpa_warning", "composite_risk"]]
print(prof.to_string())

## Q7: "Did you leak future information? How do we know the model will hold up over time?"

We avoid leakage by design: features use only data available at prediction time (e.g. mid-term GPA, attendance to date), and we use a **time-aware or stratified train/validation split** so the test set is not used in tuning. For deployment, we retrain on a schedule (e.g. weekly) or when performance drops; run metadata records each training so we can compare metrics over time and roll back if needed.

## Q8: "Why tree models (XGBoost/LightGBM) and not deep learning?"

For this problem, tree-based ensembles are the right choice: (1) **Interpretability** — SHAP and feature importance are native and stable; (2) **Sample size** — we have thousands to tens of thousands of rows per partner, not millions of high-dimensional samples where deep nets shine; (3) **Tabular data** — gradient boosting typically outperforms neural nets on tabular targets in practice; (4) **Operational simplicity** — fast inference, no GPU, and easy versioning. We use an XGBoost–LightGBM ensemble where it improves calibration; the same pipeline could swap in another estimator if requirements change.

## Q9: "What about student privacy and FERPA?"

Scores and reasons are used to **prioritize outreach**, not to automate adverse decisions. We use only data that partners have authorized for analytics; model outputs are surfaced in Salesforce (or equivalent) with the same access controls as other student records. Run metadata and artifact versioning support auditability. For net-new partners, we follow the same config-driven pipeline so data stays in their environment or under agreed governance.

## Q6: "How do we prioritize interventions? What's the ROI?"

**ABCD bands** map score ranges to actions: Band A (Critical) gets the most intensive outreach; Band D gets monitor-only. The band definitions align staff capacity with risk and are configurable per partner.

### Lead Score 1–100 with Reasons

For **prioritization**, each lead gets a **score from 1 (cold) to 100 (hot)** and a list of **reasons** (indicators that drove the score): form submit, brochure download, high engagement, responsive to outreach, etc. This makes the model actionable for enrollment teams.

In [ ]:
from src.insights import build_lead_scores_with_reasons
from src.feature_engineering import LeadScoringFeatureEngineer
from src.models import LeadScoringModel

# Ensure lead_feat and lead_model exist (run Load data + Train/Load section first, or build/load here)
_g = globals()
if "lead_feat" not in _g or "lead_model" not in _g:
    fe_lead = LeadScoringFeatureEngineer()
    merged_lead = fe_lead.merge_sources(ga4_df, crm_df, sis_df)
    lead_feat = fe_lead.create_features(merged_lead)
    lead_cols = fe_lead.get_feature_columns()
    lead_path = PROJECT_ROOT / "models" / "lead_scoring_model.pkl"
    if lead_path.exists():
        lead_model = LeadScoringModel.load(str(lead_path))
    else:
        X_lead = lead_feat[lead_cols].fillna(lead_feat[lead_cols].median()).fillna(0)
        y_lead = lead_feat["enrolled"]
        lead_model = LeadScoringModel()
        lead_model.train(X_lead, y_lead)
        lead_model.save(str(lead_path))

# Score all leads (1-100) with reasons
X_lead_full = lead_feat[lead_model.feature_names].fillna(lead_feat[lead_model.feature_names].median()).fillna(0)
lead_probs_all = lead_model.predict(X_lead_full)
lead_scores_with_reasons = build_lead_scores_with_reasons(lead_feat, lead_probs_all, limit=50)

# Display top leads by score
scores_df = pd.DataFrame(lead_scores_with_reasons)
scores_df["reasons_str"] = scores_df["reasons"].apply(lambda r: "; ".join(r[:4]) if r else "")
display(scores_df[["lead_id", "score_1_100", "band", "band_label", "enrolled", "reasons_str"]].head(20))

### What makes a lead hot? — Top indicators (from high-scoring leads)

Aggregating reasons across high-scoring leads shows which behaviors (form submit, brochure download, engagement, etc.) most often accompany "hot" leads.

In [ ]:
from collections import Counter

# Aggregate reasons from high-scoring leads (e.g. score >= 60 or band A/B)
high = [r for r in lead_scores_with_reasons if r.get("score_1_100", 0) >= 60]
lead_reason_counts = Counter()
for row in high:
    for reason in (row.get("reasons") or []):
        lead_reason_counts[reason] += 1
top_lead_reasons = lead_reason_counts.most_common(12)
labels_lead_r = [r[0] for r in top_lead_reasons]
counts_lead_r = [r[1] for r in top_lead_reasons]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(labels_lead_r))
ax.barh(y_pos, counts_lead_r, color=PALETTE["accent"], edgecolor="white", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_lead_r, fontsize=10)
ax.set_xlabel("Number of high-scoring leads with this indicator")
ax.set_title("What makes a lead hot? Top reasons (score ≥ 60)")
for i, c in enumerate(counts_lead_r):
    ax.text(c + 0.5, i, f"{c}", va="center", fontsize=9, fontweight="500")
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
BANDS = [
    ("A", 0.7, 1.0, "Critical", "Phone + meeting + academic plan"),
    ("B", 0.5, 0.7, "High", "Phone + advisor outreach"),
    ("C", 0.3, 0.5, "Medium", "Email + flag for advisor"),
    ("D", 0.0, 0.3, "Low", "Monitor only"),
]
preds = first_sem["pred_risk"]
band_counts = {}
for b, lo, hi, _, _ in BANDS:
    band_counts[b] = (preds >= lo).sum() if b == "A" else ((preds >= lo) & (preds < hi)).sum()

fig, ax = plt.subplots(figsize=(10, 4))
band_colors = [PALETTE["danger"], PALETTE["accent"], "#eab308", PALETTE["success"]]
x_pos = np.arange(len(BANDS))
width = 0.6
for i, (b, lo, hi, label, action) in enumerate(BANDS):
    cnt = band_counts.get(b, 0)
    pct = cnt / len(preds) * 100 if len(preds) > 0 else 0
    ax.bar(x_pos[i], cnt, width, color=band_colors[i], edgecolor="white", linewidth=0.5)
    ax.text(x_pos[i], cnt + 30, f"{pct:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="500")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{b[0]} ({b[3]})" for b in BANDS])
ax.set_ylabel("Number of Students")
ax.set_title("Q6: Intervention Prioritization — Band Distribution", fontsize=13, fontweight="600")
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

print("\nBand definitions:")
for b, lo, hi, label, action in BANDS:
    print(f"  {b}: {label} — {action}")

## End state: scores in Salesforce for every lead and student

**Stakeholders wanted the end result where their teams already work: the Salesforce dashboard.** Closing the loop from model → API → CRM means enrollment and success teams see **lead score**, **retention risk**, and **reasons** on each Lead and Contact record—no separate tool, no export.

| Audience | What syncs to Salesforce | Why it matters |
|----------|-------------------------|----------------|
| **Admissions / Sales** | Per **Lead**: `Lead_Score_1_100__c`, `Lead_Band__c` (A–D), `Score_Reasons__c` (top indicators) | Prioritize outreach; hot leads get callbacks first; reasons explain *why* the score is high so reps can personalize. |
| **Student Success / Advising** | Per **Contact** (student): `Retention_Risk_Prob__c`, `Risk_Band__c`, `Risk_Reasons__c`; **Coach List** (at-risk per school) as List View or report | Advisors see risk and reasons on the student record; coach list = actionable “call these students this week.” |

**Flow:** Daily score job runs (merge → features → model predict); API serves `/api/lead-scores` and `/api/coach-list` (and per-record predict); a scheduled sync (MuleSoft, Salesforce Connect, or custom ETL) pushes scores and reasons into custom fields. Same pipeline, same API—Salesforce is just the final destination so the business gets one place for all decisions.

In [ ]:
# What each lead/student sees in Salesforce — visual summary
sf_lead_fields = pd.DataFrame([
    {"Object": "Lead", "Field (example)": "Lead_Score_1_100__c", "Description": "Score 1–100 from enrollment model"},
    {"Object": "Lead", "Field (example)": "Lead_Band__c", "Description": "A=Hot, B=Warm, C=Cool, D=Cold"},
    {"Object": "Lead", "Field (example)": "Score_Reasons__c", "Description": "Top indicators (form submit, brochure, engagement…)"},
])
sf_student_fields = pd.DataFrame([
    {"Object": "Contact/Student", "Field (example)": "Retention_Risk_Prob__c", "Description": "Withdrawal probability 0–1"},
    {"Object": "Contact/Student", "Field (example)": "Risk_Band__c", "Description": "A=Critical, B=High, C=Medium, D=Low"},
    {"Object": "Contact/Student", "Field (example)": "Risk_Reasons__c", "Description": "Top drivers (GPA warning, attendance, financial…)"},
])
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].set_axis_off()
axes[0].table(cellText=sf_lead_fields.values, colLabels=sf_lead_fields.columns, loc="center", cellLoc="left", colColours=[PALETTE["teal"], PALETTE["teal"], PALETTE["teal"]])
axes[0].set_title("Leads (Admissions)", fontsize=12, fontweight="600")
axes[1].set_axis_off()
axes[1].table(cellText=sf_student_fields.values, colLabels=sf_student_fields.columns, loc="center", cellLoc="left", colColours=[PALETTE["accent"], PALETTE["accent"], PALETTE["accent"]])
axes[1].set_title("Students (Success / Coach list)", fontsize=12, fontweight="600")
plt.suptitle("Salesforce: custom fields populated from this pipeline", y=1.02, fontsize=13, fontweight="600")
plt.tight_layout()
plt.show()

## Lead Scoring — Same Rigor, Different Use Case

Enrollment prediction uses GA4 + CRM + SIS with **70% CRM join coverage** and **15% enrollment rate**. Same validation approach: calibration, SHAP, threshold analysis.

In [ ]:
# Lead model: get predictions for demo
if lead_results is None:
    from sklearn.model_selection import train_test_split
    X_lt, X_lte, y_lt, y_lte = train_test_split(X_lead, y_lead, test_size=0.2, random_state=42, stratify=y_lead)
    if hasattr(lead_model, "lgb_model") and lead_model.lgb_model is not None:
        xgb_p = lead_model.model.predict_proba(X_lte[lead_model.feature_names])[:, 1]
        lgb_p = lead_model.lgb_model.predict_proba(X_lte[lead_model.feature_names])[:, 1]
        lead_preds = 0.6 * xgb_p + 0.4 * lgb_p
    else:
        lead_preds = lead_model.predict(X_lte[lead_model.feature_names])
    lead_results = {"y_test": y_lte.values, "y_pred_proba": lead_preds}
    print(f"Lead Scoring AUC: {roc_auc_score(lead_results['y_test'], lead_results['y_pred_proba']):.3f}")

---
## Summary — Key takeaways for stakeholders

This notebook walked from data to delivery. In short:

1. **Accuracy:** AUC 78–87%; calibration aligns predicted probabilities with actual outcomes.
2. **Explainability:** SHAP and feature importance quantify what drives each prediction.
3. **Fairness:** Model behavior across first-gen and financial-aid groups is evaluated and monitored.
4. **Threshold:** The business sets the operating cutoff from precision/recall trade-offs (FP vs FN cost).
5. **Prioritization:** ABCD bands map scores to intervention intensity; configurable per partner.
6. **Data quality:** Missing exit dates, sparse joins, and class imbalance are handled explicitly in the pipeline.
7. **Validation:** No leakage (time-aware splits); tree models chosen for interpretability, sample size, and operational simplicity; retraining and run metadata support drift and rollback.
8. **Privacy:** Scores support prioritization, not automated adverse decisions; data use and access follow partner governance.
9. **Delivery:** Scores and reasons sync to **Salesforce** for each lead and student so enrollment and success teams have one place for decisions.